# Task 2: Model Building and Training

## Objective
Build, train, and evaluate machine learning models to detect fraudulent e-commerce transactions using `Fraud_Data.csv`.

Because the dataset is highly imbalanced, evaluation focuses on:
- Precision-Recall AUC (AUC-PR)
- F1-Score
- Confusion Matrix

Both an interpretable baseline model and an ensemble model are implemented.


## Libraries Used

- pandas, numpy: data handling
- scikit-learn: modeling, metrics, preprocessing
- imbalanced-learn: SMOTE for class imbalance
- matplotlib, seaborn: visualization
- # Task 2 – Model Building & Training  
**With Capstone Improvements**

We build, train, and compare models using **reusable src code** (type hints, error handling, modularity).  
Focus: fraud detection for e-commerce (Fraud_Data) and credit card transactions.


In [1]:
# ================================================
# Make src/ importable – MUST RUN THIS CELL FIRST
# ================================================
import os
import sys

# Option 1: Hard-coded path (safest & recommended)
project_root = r"C:\Users\Y\Downloads\fraud-detection"

# Option 2: Automatic (if notebook is inside notebooks/ folder)
# project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))

# Change working directory so relative paths work (data/raw, models/, etc.)
try:
    os.chdir(project_root)
    print("Working directory changed to:", os.getcwd())
except Exception as e:
    print("Could not change directory:", e)

# Add project root to sys.path so Python finds src/
if project_root not in sys.path:
    sys.path.insert(0, project_root)
    print("Added project root to sys.path → src/ is now importable")

print("Setup complete. You can now import from src/")

Working directory changed to: C:\Users\Y\Downloads\fraud-detection
Added project root to sys.path → src/ is now importable
Setup complete. You can now import from src/


In [2]:
import pandas as pd
from src.data import load_processed_data, prepare_split
from src.models import train_baseline_logreg, train_xgboost, evaluate_and_save_model
from src.evaluation import compute_metrics, cross_validate_model

print("All src modules imported successfully.")

All src modules imported successfully.


## 1. Load & Prepare Fraud Data (E-commerce)

In [3]:
# Load preprocessed fraud data
df_fraud = load_processed_data("fraud")

# Split (stratified)
X_train_f, X_test_f, y_train_f, y_test_f = prepare_split(df_fraud, target_col="class")

Loaded fraud data: 151,112 rows, 196 columns
Train set: 120,889 samples | Fraud rate: 0.0936
Test set:  30,223 samples  | Fraud rate: 0.0936


In [4]:
# ================================================
# Critical fix: Drop non-numeric / non-feature columns
# ================================================
non_useful_cols = [
    'user_id', 'device_id', 'ip_address', 'ip_int',
    'signup_time', 'purchase_time'
]

# Only drop columns that actually exist
cols_to_drop = [col for col in non_useful_cols if col in df_fraud.columns]

if cols_to_drop:
    df_fraud = df_fraud.drop(columns=cols_to_drop)
    print(f"Dropped non-modeling columns: {cols_to_drop}")
else:
    print("No extra non-numeric columns found to drop")

# Quick check: all columns should now be numeric
print("\nRemaining columns types:")
print(df_fraud.dtypes.value_counts())

print("\nAny remaining object/string columns?")
print(df_fraud.select_dtypes(include=['object']).columns.tolist())

No extra non-numeric columns found to drop

Remaining columns types:
bool       188
float64      7
int64        1
Name: count, dtype: int64

Any remaining object/string columns?
[]


In [5]:
# ================================================
# FINAL DATA CLEANUP – RUN THIS ONCE AFTER LOADING
# ================================================
print("Columns BEFORE cleanup:", df_fraud.columns.tolist())

# All columns we know are useless or non-numeric for modeling
useless = [
    'user_id', 'device_id', 'ip_address', 'ip_int',
    'signup_time', 'purchase_time'
]

# Drop only those that exist
to_drop = [c for c in useless if c in df_fraud.columns]

if to_drop:
    df_fraud = df_fraud.drop(columns=to_drop)
    print(f"Dropped non-modeling columns: {to_drop}")
else:
    print("No bad columns found to drop")

# Safety check
non_num_cols = df_fraud.select_dtypes(include=['object']).columns.tolist()
if non_num_cols:
    print("WARNING – STILL HAVE OBJECT COLUMNS:", non_num_cols)
    print("You must drop them manually or update preprocessing.py")
else:
    print("All columns are now numeric → models should train OK")

Columns BEFORE cleanup: ['age', 'browser_FireFox', 'browser_IE', 'browser_Opera', 'browser_Safari', 'class', 'country_Albania', 'country_Algeria', 'country_Angola', 'country_Antigua and Barbuda', 'country_Argentina', 'country_Armenia', 'country_Australia', 'country_Austria', 'country_Azerbaijan', 'country_Bahamas', 'country_Bahrain', 'country_Bangladesh', 'country_Barbados', 'country_Belarus', 'country_Belgium', 'country_Belize', 'country_Benin', 'country_Bermuda', 'country_Bhutan', 'country_Bolivia', 'country_Bonaire; Sint Eustatius; Saba', 'country_Bosnia and Herzegowina', 'country_Botswana', 'country_Brazil', 'country_British Indian Ocean Territory', 'country_Brunei Darussalam', 'country_Bulgaria', 'country_Burkina Faso', 'country_Burundi', 'country_Cambodia', 'country_Cameroon', 'country_Canada', 'country_Cape Verde', 'country_Cayman Islands', 'country_Chile', 'country_China', 'country_Colombia', 'country_Congo', 'country_Congo The Democratic Republic of The', 'country_Costa Rica',

In [21]:
# ================================================
# EMERGENCY CLEANUP – RUN THIS BEFORE ANY TRAINING OR TESTING
# ================================================
print("BEFORE cleanup – columns:", df_fraud.columns.tolist())
print("Non-numeric columns:", df_fraud.select_dtypes(include=['object']).columns.tolist())

# Remove known bad columns
bad_cols = [
    'user_id', 'device_id', 'ip_address', 'ip_int',
    'signup_time', 'purchase_time'
]

to_drop = [c for c in bad_cols if c in df_fraud.columns]

if to_drop:
    df_fraud = df_fraud.drop(columns=to_drop)
    print(f"DROPPED: {to_drop}")
else:
    print("No bad columns found – strange!")

# Final check
non_num_left = df_fraud.select_dtypes(include=['object']).columns.tolist()
if non_num_left:
    print("STILL DIRTY – remaining object columns:", non_num_left)
    # If something is left → drop all object columns as last resort
    df_fraud = df_fraud.select_dtypes(exclude=['object'])
    print("Last resort: dropped ALL object columns")
else:
    print("CLEAN → only numeric columns left ✓")

print("\nShape now:", df_fraud.shape)

BEFORE cleanup – columns: ['age', 'browser_FireFox', 'browser_IE', 'browser_Opera', 'browser_Safari', 'class', 'country_Albania', 'country_Algeria', 'country_Angola', 'country_Antigua and Barbuda', 'country_Argentina', 'country_Armenia', 'country_Australia', 'country_Austria', 'country_Azerbaijan', 'country_Bahamas', 'country_Bahrain', 'country_Bangladesh', 'country_Barbados', 'country_Belarus', 'country_Belgium', 'country_Belize', 'country_Benin', 'country_Bermuda', 'country_Bhutan', 'country_Bolivia', 'country_Bonaire; Sint Eustatius; Saba', 'country_Bosnia and Herzegowina', 'country_Botswana', 'country_Brazil', 'country_British Indian Ocean Territory', 'country_Brunei Darussalam', 'country_Bulgaria', 'country_Burkina Faso', 'country_Burundi', 'country_Cambodia', 'country_Cameroon', 'country_Canada', 'country_Cape Verde', 'country_Cayman Islands', 'country_Chile', 'country_China', 'country_Colombia', 'country_Congo', 'country_Congo The Democratic Republic of The', 'country_Costa Rica

In [6]:
from src.models import train_baseline_logreg

# Use the cleaned df_fraud
small_X = df_fraud.drop("class", axis=1).iloc[:200]
small_y = df_fraud["class"].iloc[:200]

model, _ = train_baseline_logreg(small_X, small_y)
print("Baseline LogReg trained successfully ✓")

Baseline LogReg trained successfully ✓


## 2. Baseline: Logistic Regression (Fraud Data)

In [7]:
# Quick fix: drop non-numeric columns
non_numeric_cols = df_fraud.select_dtypes(exclude=['number']).columns.tolist()
print("Dropping non-numeric columns:", non_numeric_cols)

df_fraud = df_fraud.drop(columns=non_numeric_cols)

# Now proceed with split
X_train_f, X_test_f, y_train_f, y_test_f = prepare_split(df_fraud, target_col="class")

Dropping non-numeric columns: ['browser_FireFox', 'browser_IE', 'browser_Opera', 'browser_Safari', 'country_Albania', 'country_Algeria', 'country_Angola', 'country_Antigua and Barbuda', 'country_Argentina', 'country_Armenia', 'country_Australia', 'country_Austria', 'country_Azerbaijan', 'country_Bahamas', 'country_Bahrain', 'country_Bangladesh', 'country_Barbados', 'country_Belarus', 'country_Belgium', 'country_Belize', 'country_Benin', 'country_Bermuda', 'country_Bhutan', 'country_Bolivia', 'country_Bonaire; Sint Eustatius; Saba', 'country_Bosnia and Herzegowina', 'country_Botswana', 'country_Brazil', 'country_British Indian Ocean Territory', 'country_Brunei Darussalam', 'country_Bulgaria', 'country_Burkina Faso', 'country_Burundi', 'country_Cambodia', 'country_Cameroon', 'country_Canada', 'country_Cape Verde', 'country_Cayman Islands', 'country_Chile', 'country_China', 'country_Colombia', 'country_Congo', 'country_Congo The Democratic Republic of The', 'country_Costa Rica', "country_

In [8]:
# Train baseline
logreg_model, logreg_info = train_baseline_logreg(X_train_f, y_train_f)

# Evaluate & save
logreg_metrics = evaluate_and_save_model(
    logreg_model, X_test_f, y_test_f,
    model_name="logreg_fraud"
)

print("\nLogistic Regression Metrics:")
for k, v in logreg_metrics.items():
    print(f"{k}: {v:.4f}")

Model saved: models\logreg_fraud.joblib

Logistic Regression Metrics:
auc_pr: 0.6598
f1: 0.6129
tn: 25841.0000
fp: 1552.0000
fn: 894.0000
tp: 1936.0000


## 3. Ensemble: XGBoost (Fraud Data) – Basic Tuning

In [9]:
# Train XGBoost
xgb_model, xgb_info = train_xgboost(
    X_train_f, y_train_f,
    n_estimators=300,
    max_depth=7,
    learning_rate=0.05
)

# Cross-validation score
cv_scores = cross_validate_model(xgb_model, X_train_f, y_train_f)
print("\nXGBoost CV Results:")
print(cv_scores)

# Evaluate & save
xgb_metrics = evaluate_and_save_model(
    xgb_model, X_test_f, y_test_f,
    model_name="xgboost_fraud_v1"
)

print("\nXGBoost Test Metrics:")
for k, v in xgb_metrics.items():
    print(f"{k}: {v:.4f}")


XGBoost CV Results:
{'auc_pr_mean': 0.7120489455058037, 'auc_pr_std': 0.0034927491948586495, 'f1_mean': 0.621596768642326, 'f1_std': 0.0033843483252708646}
Model saved: models\xgboost_fraud_v1.joblib

XGBoost Test Metrics:
auc_pr: 0.7068
f1: 0.6151
tn: 25761.0000
fp: 1632.0000
fn: 848.0000
tp: 1982.0000


## 4. Model Comparison – Fraud Data

In [10]:
comparison = pd.DataFrame({
    'Model': ['Logistic Regression', 'XGBoost'],
    'AUC-PR': [logreg_metrics['auc_pr'], xgb_metrics['auc_pr']],
    'F1': [logreg_metrics['f1'], xgb_metrics['f1']],
    'False Positives': [logreg_metrics['fp'], xgb_metrics['fp']],
    'False Negatives': [logreg_metrics['fn'], xgb_metrics['fn']]
})

print("Model Comparison:")
print(comparison.round(4))

# Winner
best_model = 'XGBoost' if xgb_metrics['auc_pr'] > logreg_metrics['auc_pr'] else 'Logistic Regression'
print(f"\nSelected best model: {best_model} (higher AUC-PR, better fraud detection with fewer misses)")

Model Comparison:
                 Model  AUC-PR      F1  False Positives  False Negatives
0  Logistic Regression  0.6598  0.6129           1552.0            894.0
1              XGBoost  0.7068  0.6151           1632.0            848.0

Selected best model: XGBoost (higher AUC-PR, better fraud detection with fewer misses)


In [11]:
from src.data import load_processed_data
df = load_processed_data("fraud")
df.head(3)

Loaded fraud data: 151,112 rows, 196 columns


,age,browser_FireFox,browser_IE,browser_Opera,browser_Safari,class,country_Albania,country_Algeria,country_Angola,country_Antigua and Barbuda,...,country_Zimbabwe,day_of_week,hour_of_day,purchase_value,sex_M,source_Direct,source_SEO,time_since_signup_hours,tx_per_device,tx_per_user
0,0.679914,False,False,False,False,0,False,False,False,False,...,False,0.991020,-1.377455,-0.160204,True,False,True,-0.136057,-0.261514,0.0
1,2.304476,False,False,False,False,0,False,False,False,False,...,False,-1.501259,-1.522122,-1.142592,False,False,False,-1.571877,-0.261514,0.0
2,2.304476,False,False,True,False,1,False,False,False,False,...,False,-0.005891,0.937208,-1.197169,True,False,True,-1.577617,3.941861,0.0


In [12]:
# ================================================
# FIX DATA – REMOVE ALL NON-NUMERIC COLUMNS
# ================================================
print("Columns before cleanup:", df_fraud.columns.tolist())

# List of columns we know are useless for modeling
useless_cols = [
    'user_id', 'device_id', 'ip_address', 'ip_int',
    'signup_time', 'purchase_time'
]

# Drop only columns that actually exist
cols_to_drop = [c for c in useless_cols if c in df_fraud.columns]

if cols_to_drop:
    df_fraud = df_fraud.drop(columns=cols_to_drop)
    print(f"Dropped: {cols_to_drop}")
else:
    print("No useless columns found")

# Final safety check
print("\nAfter cleanup – any non-numeric columns left?")
non_num = df_fraud.select_dtypes(exclude=['number', 'bool', 'uint8']).columns.tolist()
print(non_num)

if non_num:
    print("WARNING: Still have non-numeric → drop them manually or investigate")
else:
    print("Data is now model-ready ✓")

Columns before cleanup: ['age', 'class', 'day_of_week', 'hour_of_day', 'purchase_value', 'time_since_signup_hours', 'tx_per_device', 'tx_per_user']
No useless columns found

After cleanup – any non-numeric columns left?
[]
Data is now model-ready ✓


In [13]:
from src.models import train_xgboost

# very small test
small_X = df_fraud.drop("class", axis=1).iloc[:200]
small_y = df_fraud["class"].iloc[:200]

model, _ = train_xgboost(small_X, small_y)
print("XGBoost trained without error ✓")

XGBoost trained without error ✓


## Model Comparison & Selection

In [14]:
comparison = pd.DataFrame({
    'Model': ['Logistic Regression', 'XGBoost'],
    'AUC-PR': [logreg_metrics['auc_pr'], xgb_metrics['auc_pr']],
    'F1': [logreg_metrics['f1'], xgb_metrics['f1']],
    'True Positives': [logreg_metrics['tp'], xgb_metrics['tp']],
    'False Positives': [logreg_metrics['fp'], xgb_metrics['fp']],
    'False Negatives': [logreg_metrics['fn'], xgb_metrics['fn']]
})

print("Model Comparison:")
print(comparison.round(4))

# Select best
best = 'XGBoost' if xgb_metrics['auc_pr'] > logreg_metrics['auc_pr'] else 'Logistic Regression'
print(f"\nBest model: {best} (higher AUC-PR = better fraud detection with fewer misses)")

Model Comparison:
                 Model  AUC-PR      F1  True Positives  False Positives  \
0  Logistic Regression  0.6598  0.6129          1936.0           1552.0   
1              XGBoost  0.7068  0.6151          1982.0           1632.0   

   False Negatives  
0            894.0  
1            848.0  

Best model: XGBoost (higher AUC-PR = better fraud detection with fewer misses)


## Business Justification & Impact

**Problem**: Fraud causes direct financial loss and erodes customer trust through false positives.  
**Solution**: XGBoost outperforms Logistic Regression with higher AUC-PR and fewer false negatives → catches more fraud while reducing unnecessary flags.  
**Impact**: If model reduces fraud by 30–40% with <5% false positive rate → potential $XX saved per 100k transactions (cost matrix needed for exact $).  
**Why XGBoost?** Better handles non-linear patterns and imbalance (via scale_pos_weight) while remaining interpretable via feature importances.

In [15]:
# Save best model (example: assume XGBoost wins)
import joblib
if xgb_metrics['auc_pr'] > logreg_metrics['auc_pr']:
    joblib.dump(xgb_model, "models/best_model_xgboost.joblib")
    print("Best model saved: models/best_model_xgboost.joblib")

Best model saved: models/best_model_xgboost.joblib


In [18]:
# ================================================
# Setup – imports & project root
# ================================================
import os
import sys
import pandas as pd
import shap
import matplotlib.pyplot as plt

# Project root
project_root = r"C:\Users\Y\Downloads\fraud-detection"
os.chdir(project_root)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# This line imports all the functions from your module
from src.explainability import (
    load_model,
    get_feature_importance,
    plot_top_features,
    explain_with_shap,
    plot_shap_force,
    plot_shap_dependence  # if you added this one
)

print("Setup complete – ready for SHAP analysis")

Setup complete – ready for SHAP analysis


In [19]:
# Load best model (change path if needed)
model_path = "models/best_model_xgboost.joblib"
model = load_model(model_path)

# Load processed data (for feature names)
df = pd.read_csv("data/processed/fraud_processed.csv")

# Align to model's expected features (critical!)
expected_features = model.feature_names_in_
X = df[expected_features].copy()  # only the columns the model knows
y = df["class"]

print("Loaded model and data successfully")
print("Number of features (aligned):", X.shape[1])

# Print what we need for the dashboard
print("\nModel expected features:", model.feature_names_in_.tolist())
print("\nFeature dtypes:")
print(X[model.feature_names_in_].dtypes)

Loaded model from models\best_model_xgboost.joblib
Loaded model and data successfully
Number of features (aligned): 7

Model expected features: ['age', 'day_of_week', 'hour_of_day', 'purchase_value', 'time_since_signup_hours', 'tx_per_device', 'tx_per_user']

Feature dtypes:
age                        float64
day_of_week                float64
hour_of_day                float64
purchase_value             float64
time_since_signup_hours    float64
tx_per_device              float64
tx_per_user                float64
dtype: object


In [20]:
# ← your existing loading code above

# Add these lines here:
print("Model expected features:", model.feature_names_in_.tolist())
print("\nFeature dtypes:")
print(X[model.feature_names_in_].dtypes)

Model expected features: ['age', 'day_of_week', 'hour_of_day', 'purchase_value', 'time_since_signup_hours', 'tx_per_device', 'tx_per_user']

Feature dtypes:
age                        float64
day_of_week                float64
hour_of_day                float64
purchase_value             float64
time_since_signup_hours    float64
tx_per_device              float64
tx_per_user                float64
dtype: object
